In [1]:
import pandas as pd
import numpy as np 
import matplotlib as plt
import sklearn as sl
import os

In [2]:
#Exporting the data
folder_path = r"C:\Users\USER\Desktop\final readings\2.5 feet readings"  # your folder path

all_data = []

for file in os.listdir(folder_path):
    if file.endswith(".xlsx"):
        file_path = os.path.join(folder_path, file)
        df = pd.read_excel(file_path)
        all_data.append(df)

# Combine all files into one DataFrame
combined_df = pd.concat(all_data, ignore_index=True)

print(combined_df.shape)

(3626733, 8)


In [3]:
combined_df.describe()

,Frequency(Hz),Magnitude(db),Phase(degrees),Angle of turn table,x,y,z,Distance
count,3.626733e+06,3.626733e+06,3.626733e+06,3.626733e+06,3.626733e+06,3.626733e+06,3.626733e+06,3.373854e+06
mean,1.000001e+10,2.154004e+03,1.193002e+00,1.699559e+02,1.477493e+00,5.483672e+00,1.177539e+00,6.087930e+00
std,1.155070e+09,4.200802e+06,1.031108e+02,1.037977e+02,1.295207e+00,3.641049e+00,3.596365e-01,3.348404e+00
min,8.000000e+09,-1.229500e+02,-1.800000e+02,0.000000e+00,0.000000e+00,0.000000e+00,8.000000e-01,9.433981e-01
25%,9.000000e+09,-6.115240e+01,-8.731640e+01,8.000000e+01,0.000000e+00,2.000000e+00,8.000000e-01,2.939388e+00
50%,1.000000e+10,-5.231510e+01,-7.056550e-03,1.600000e+02,1.000000e+00,6.000000e+00,1.520000e+00,6.053098e+00
75%,1.100000e+10,-4.318792e+01,8.980550e+01,2.600000e+02,3.000000e+00,9.000000e+00,1.520000e+00,9.035486e+00
max,1.200000e+10,8.000000e+09,1.800000e+02,3.400000e+02,4.000000e+00,1.100000e+01,1.524000e+00,1.150263e+01


In [4]:
combined_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3626733 entries, 0 to 3626732
Data columns (total 8 columns):
 #   Column               Dtype  
---  ------               -----  
 0   Frequency(Hz)        int64  
 1   Magnitude(db)        float64
 2   Phase(degrees)       float64
 3   Angle of turn table  int64  
 4   x                    int64  
 5   y                    float64
 6   z                    float64
 7   Distance             float64
dtypes: float64(5), int64(3)
memory usage: 221.4 MB


In [5]:
#Deleting Extra columns

data= combined_df.drop('Distance', axis=1)

In [6]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 3626733 entries, 0 to 3626732
Data columns (total 7 columns):
 #   Column               Dtype  
---  ------               -----  
 0   Frequency(Hz)        int64  
 1   Magnitude(db)        float64
 2   Phase(degrees)       float64
 3   Angle of turn table  int64  
 4   x                    int64  
 5   y                    float64
 6   z                    float64
dtypes: float64(4), int64(3)
memory usage: 193.7 MB


In [7]:
# Checking for mising and null values
#data.isna()
#data.isnull()
print(data.isnull().sum())
#data[data.isnull().any(axis=1)]

Frequency(Hz)          0
Magnitude(db)          0
Phase(degrees)         0
Angle of turn table    0
x                      0
y                      0
z                      0
dtype: int64


In [ ]:
# Removing Null values
data=data.dropna()

In [8]:
#converting Magnitude and Phase into Complex form
data['Magnitude_Lin']=10**(data['Magnitude(db)']/20)
data['Phase_rad']=np.deg2rad(data['Phase(degrees)'])
data['complex']=data['Magnitude_Lin']*np.exp(1j*data['Phase_rad'])

In [9]:
data.drop(columns=['Magnitude_Lin','Phase_rad'], inplace=True)
data.head()

,Frequency(Hz),Magnitude(db),Phase(degrees),Angle of turn table,x,y,z,complex
0,8000000000,-22.42076,-70.99882,0,0,1.0,1.52,0.024639-0.071553j
1,8001250000,-22.70777,-69.50725,0,0,1.0,1.52,0.025632-0.068584j
2,8002500000,-22.54431,-75.13818,0,0,1.0,1.52,0.019136-0.072112j
3,8003750000,-23.10117,-79.66623,0,0,1.0,1.52,0.012552-0.068840j
4,8005000000,-22.74971,-87.64665,0,0,1.0,1.52,0.002992-0.072803j


In [ ]:
# Few checks
# Check complex column exists and is correct type
print(data['complex'].dtype)       # should be complex128
print(data['complex'].head())

# Check phase range — should stay within [-180, 180]
print(data['Phase(degrees)'].min(), data['Phase(degrees)'].max())

# Check how many unique positions you have
print(data.groupby(['x', 'y', 'z', 'Angle of turn table']).ngroups)

# Check frequency sweep per position
print(data.groupby(['x', 'y', 'z', 'Angle of turn table']).size().unique())
# Should return [3201] — confirming 3201 freq points per position

print(len(data)) 

complex128
0    0.024639-0.071553j
1    0.025632-0.068584j
2    0.019136-0.072112j
3    0.012552-0.068840j
4    0.002992-0.072803j
Name: complex, dtype: complex128
-180.0 180.0
1133
[3201]
3626733


In [11]:
counts = data.groupby(['x', 'y', 'z', 'Angle of turn table']).size()

print("=== Duplicates (6402) ===")
print(counts[counts == 6402])

print("=== Incomplete (3200) ===")
print(counts[counts == 3200])

print("=== Partial (1848) ===")
print(counts[counts == 1848])

print("=== Single point (1) ===")
print(counts[counts == 1])

=== Duplicates (6402) ===
Series([], dtype: int64)
=== Incomplete (3200) ===
Series([], dtype: int64)
=== Partial (1848) ===
Series([], dtype: int64)
=== Single point (1) ===
Series([], dtype: int64)


In [ ]:
# For each z value, show which (x,y) positions exist
for z_val in sorted(data['z'].unique()):
    positions = data[data['z'] == z_val][['x', 'y']].drop_duplicates().sort_values(['x', 'y'])
    print(f"\n{'='*40}")
    print(f"z = {z_val} → {len(positions)} unique (x,y) positions")
    print(positions.to_string(index=False))

In [12]:
conflict = data[(data['x']==1.0) & (data['y']==1.0) & (data['z']==1.52)]
print(f"Rows at (x=1, y=1, z=1.52): {len(conflict)}")

conflict2 = data[(data['x']==1.0) & (data['y']==1.0) & (data['z']==1.524)]
print(f"Rows at (x=1, y=1, z=1.524): {len(conflict2)}")

Rows at (x=1, y=1, z=1.52): 0
Rows at (x=1, y=1, z=1.524): 57618


In [ ]:
data['z'] = data['z'].replace(1.524, 1.52)
print(data['z'].unique())  # will show [0.8, 1.52, 3.0]

[1.52 0.8 ]


In [14]:
# Check for exact duplicate rows across all columns
n_before = len(data)
n_dupes = data.duplicated().sum()
print(f"Exact duplicate rows: {n_dupes}")

# Check duplicates on key columns only (excluding complex which may differ)
key_cols = ['Frequency(Hz)', 'x', 'y', 'z', 'Angle of turn table']
n_key_dupes = data.duplicated(subset=key_cols).sum()
print(f"Duplicate rows on key columns: {n_key_dupes}")

Exact duplicate rows: 0
Duplicate rows on key columns: 0


In [15]:
counts = data.groupby(['x', 'y', 'z', 'Angle of turn table']).size()

print("=== FINAL DATASET STATUS ===")
print(f"Group size distribution : {sorted(counts.unique())}")
print(f"Total unique positions  : {len(counts)}")
print(f"Total rows              : {len(data)}")
print(f"Expected rows           : {len(counts)} × 3201 = {len(counts)*3201}")
print(f"z values                : {sorted(data['z'].unique())}")
print(f"Angle values            : {sorted(data['Angle of turn table'].unique())}")
print(f"Frequency range         : {data['Frequency(Hz)'].min():.4e} to {df['Frequency(Hz)'].max():.4e}")
print(f"Exact duplicates        : {data.duplicated().sum()}")
print(f"Key column duplicates   : {data.duplicated(subset=['Frequency(Hz)','x','y','z','Angle of turn table']).sum()}")
print(f"NaN values              : {data.isna().sum().sum()}")

=== FINAL DATASET STATUS ===
Group size distribution : [np.int64(3201)]
Total unique positions  : 1133
Total rows              : 3626733
Expected rows           : 1133 × 3201 = 3626733
z values                : [np.float64(0.8), np.float64(1.52)]
Angle values            : [np.int64(0), np.int64(20), np.int64(40), np.int64(60), np.int64(80), np.int64(100), np.int64(120), np.int64(140), np.int64(160), np.int64(180), np.int64(200), np.int64(220), np.int64(240), np.int64(260), np.int64(280), np.int64(300), np.int64(320), np.int64(340)]
Frequency range         : 8.0000e+09 to 1.2000e+10
Exact duplicates        : 0
Key column duplicates   : 0
NaN values              : 0


In [ ]:
# Split complex into two real columns for saving
data['H_real'] = data['complex'].apply(lambda x: x.real)
data['H_imag'] = data['complex'].apply(lambda x: x.imag)

# Drop the complex128 column before saving
data_save = data.drop(columns=['complex'])

In [23]:
# Now save as csv
data_save.to_csv('CTF_clean.csv', index=False)
print(f"Saved: CTF_clean.csv")
print(f"Rows saved: {len(data_save)}")
print(f"Columns: {list(data_save.columns)}")

Saved: CTF_clean.csv
Rows saved: 3626733
Columns: ['Frequency(Hz)', 'Magnitude(db)', 'Phase(degrees)', 'Angle of turn table', 'x', 'y', 'z', 'H_real', 'H_imag']


In [24]:
df_check = pd.read_csv('CTF_clean.csv', nrows=5)
print(df_check.head())
print(f"Columns: {list(df_check.columns)}")

   Frequency(Hz)  Magnitude(db)  Phase(degrees)  Angle of turn table  x    y  \
0     8000000000      -22.42076       -70.99882                    0  0  1.0   
1     8001250000      -22.70777       -69.50725                    0  0  1.0   
2     8002500000      -22.54431       -75.13818                    0  0  1.0   
3     8003750000      -23.10117       -79.66623                    0  0  1.0   
4     8005000000      -22.74971       -87.64665                    0  0  1.0   

      z    H_real    H_imag  
0  1.52  0.024639 -0.071553  
1  1.52  0.025632 -0.068584  
2  1.52  0.019136 -0.072112  
3  1.52  0.012552 -0.068840  
4  1.52  0.002992 -0.072803  
Columns: ['Frequency(Hz)', 'Magnitude(db)', 'Phase(degrees)', 'Angle of turn table', 'x', 'y', 'z', 'H_real', 'H_imag']


In [ ]:
import pandas as pd
import numpy as np

data = pd.read_csv('CTF_clean.csv')

# Reconstruct complex column
data['complex'] = data['H_real'].values + 1j * data['H_imag'].values
print(f"Rows loaded: {len(data)}")
print(f"Complex dtype: {data['complex'].dtype}")